In [1]:
from openai import OpenAI
import gymnasium as gym
from tinydb import TinyDB
import os

from navigation.environments.FrozenLakeEnv import FrozenLakeEnv
from navigation.environments.FrozenLakeShadowEnv import FrozenLakeShadowEnv
from navigation.Navigator import Navigator

from optimization.prompts.FrozenLakePrompts import FrozenLakePrompts
from optimization.hypotheses.HypothesesRefiner import HypothesesRefiner
from optimization.policy.PolicyRefiner import PolicyRefiner

In [2]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key= os.getenv("OPENROUTER_API_KEY")
) 
model = "openai/gpt-oss-120b"

hypothesesDb = TinyDB("../store/frozenlake/hypotheses.json")
policyDb = TinyDB("../store/frozenlake/policies.json")

big_map = [
    "SFFFFFHFFFF",
    "FFFFFHFFFFF",
    "FFFFFFHFHFF",
    "FFFFFFFFFFF",
    "FFFFFFFFFFF",
    "FFFFFFFFFFF",
    "FFHFFFFFFFF",
    "FFFFFFFFFHF",
    "FFFFFFFFFFF",
    "FFFFFFFHFFF",
    "FFFFFFFFFFG"
]
medium_map = [
    "SFFFFHF",
    "FFFFFHF",
    "FHFFFFH",
    "FFFFFFF",
    "FFFHFFF",
    "FFHFFFG"
    ]
small_map = [
    "SFFF",
    "FFFH",
    "HFFH",
    "FHFF",
    "FFFG"
] 

In [3]:
env = FrozenLakeEnv(env = gym.make("FrozenLake-v1", render_mode="ansi", desc=small_map, map_name=None, is_slippery=True, success_rate=0.8, reward_schedule=(1, 0, 0)))
shadow_env = FrozenLakeShadowEnv(hypothesesDb, policyDb, client, model) 
env.reset()

optimizationPrompts = FrozenLakePrompts(policyDb=policyDb, hypothesesDb=hypothesesDb)

In [4]:
debug=True

for i in range(5):
    # Navigation loop to gernate trajectory
    env.reset()
    navigator = Navigator(env, shadow_env)
    trajectory = navigator.run(sample_size=2, depth=2, use_llm_action=False, debug=debug)

    # Optimization loop to refine hypotheses and strategies based on trajectory
    hypothesisRefiner = HypothesesRefiner(client, model, hypothesesDb)
    hypothesisRefiner.run(trajectory, debug=debug)

    policyRefiner = PolicyRefiner(client, model, policyDb, optimizationPrompts)
    policyRefiner.run(trajectory, debug=debug)

Sample 1, Step 1: Executed Move: move_right, Value: 85
New State:
 S [F] F  F 
 F  F  F  H 
 H  F  F  H 
 F  H  F  F 
 F  F  F  G 

Sample 1, Step 2: Executed Move: move_down, Value: 88
New State:
 S  F  F  F 
 F [F] F  H 
 H  F  F  H 
 F  H  F  F 
 F  F  F  G 

Sample 2, Step 1: Executed Move: move_right, Value: 85
New State:
 S [F] F  F 
 F  F  F  H 
 H  F  F  H 
 F  H  F  F 
 F  F  F  G 

Sample 2, Step 2: Executed Move: move_right, Value: 70
New State:
 S  F [F] F 
 F  F  F  H 
 H  F  F  H 
 F  H  F  F 
 F  F  F  G 

## Step: 1
Current State:
[S] F  F  F 
 F  F  F  H 
 H  F  F  H 
 F  H  F  F 
 F  F  F  G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_right, Value: 85
- Step 2: Move: move_down, Value: 88

Executed Move: move_right
Reward Received: 0

Sample 1, Step 1: Executed Move: move_right, Value: 85
New State:
 S  F [F] F 
 F  F  F  H 
 H  F  F  H 
 F  H  F  F 
 F  F  F  G 

Sample 1, Step 2: Executed Move: move_down, Value: 92
N

In [ ]:
debugTrajectory = '''
## Step: 1
Current State:
[S] F  F  F 
 F  H  F  H 
 F  F  F  H 
 H  F  F  G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_right, Value: 72
- Step 2: Move: move_right, Value: 75

Executed Move: move_right
Reward Received: 0

## Step: 2
Current State:
 S [F] F  F 
 F  H  F  H 
 F  F  F  H 
 H  F  F  G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_right, Value: 70
- Step 2: Move: move_down, Value: 85

Executed Move: move_right
Reward Received: 0

## Step: 3
Current State:
 S  F [F] F 
 F  H  F  H 
 F  F  F  H 
 H  F  F  G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_down, Value: 90
- Step 2: Move: move_right, Value: 85

Executed Move: move_down
Reward Received: 0

## Step: 4
Current State:
 S [F] F  F 
 F  H  F  H 
 F  F  F  H 
 H  F  F  G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_right, Value: 85
- Step 2: Move: move_down, Value: 90

Executed Move: move_right
Reward Received: 0

## Step: 5
Current State:
 S  F [F] F 
 F  H  F  H 
 F  F  F  H 
 H  F  F  G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_down, Value: 85
- Step 2: Move: move_down, Value: 85

Executed Move: move_down
Reward Received: 0

## Step: 6
Current State:
 S  F  F  F 
 F  H [F] H 
 F  F  F  H 
 H  F  F  G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_down, Value: 75

Executed Move: move_down
Reward Received: 0
Navigation was terminated.
Final State:
 S  F  F  F 
 F  H  F [H]
 F  F  F  H 
 H  F  F  G '''

debugTrajectory2='''
## Step: 1
Current State:
[S] F  F  F 
 F  H  F  H 
 F  F  F  H 
 H  F  F  G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_right, Value: 75
- Step 2: Move: move_right, Value: 85

Executed Move: move_right
Reward Received: 0

## Step: 2
Current State:
 S [F] F  F 
 F  H  F  H 
 F  F  F  H 
 H  F  F  G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_right, Value: 85
- Step 2: Move: move_down, Value: 75

Executed Move: move_right
Reward Received: 0

## Step: 3
Current State:
 S  F [F] F 
 F  H  F  H 
 F  F  F  H 
 H  F  F  G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_down, Value: 85
- Step 2: Move: move_down, Value: 75

Executed Move: move_down
Reward Received: 0

## Step: 4
Current State:
 S  F  F  F 
 F  H [F] H 
 F  F  F  H 
 H  F  F  G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_down, Value: 88

Executed Move: move_down
Reward Received: 0
Navigation was terminated.
Final State:
 S  F  F  F 
 F  H  F [H]
 F  F  F  H 
 H  F  F  G '''

debugTrajectory3='''
## Step: 1
Current State:
[S] F  F  F  F  H  F 
 F  F  F  F  F  H  F 
 F  H  F  F  F  F  H 
 F  F  F  F  F  F  F 
 F  F  F  H  F  F  F 
 F  F  H  F  F  F  G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_right, Value: 72
- Step 2: Move: move_right, Value: 72

Executed Move: move_right
Reward Received: 0

## Step: 2
Current State:
 S [F] F  F  F  H  F 
 F  F  F  F  F  H  F 
 F  H  F  F  F  F  H 
 F  F  F  F  F  F  F 
 F  F  F  H  F  F  F 
 F  F  H  F  F  F  G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_right, Value: 80
- Step 2: Move: move_down, Value: 70

Executed Move: move_right
Reward Received: 0

## Step: 3
Current State:
 S  F [F] F  F  H  F 
 F  F  F  F  F  H  F 
 F  H  F  F  F  F  H 
 F  F  F  F  F  F  F 
 F  F  F  H  F  F  F 
 F  F  H  F  F  F  G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_right, Value: 78
- Step 2: Move: move_down, Value: 75

Executed Move: move_right
Reward Received: 0

## Step: 4
Current State:
 S  F  F [F] F  H  F 
 F  F  F  F  F  H  F 
 F  H  F  F  F  F  H 
 F  F  F  F  F  F  F 
 F  F  F  H  F  F  F 
 F  F  H  F  F  F  G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_down, Value: 75
- Step 2: Move: move_down, Value: 75

Executed Move: move_down
Reward Received: 0

## Step: 5
Current State:
 S  F  F  F  F  H  F 
 F  F  F [F] F  H  F 
 F  H  F  F  F  F  H 
 F  F  F  F  F  F  F 
 F  F  F  H  F  F  F 
 F  F  H  F  F  F  G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_down, Value: 78
- Step 2: Move: move_down, Value: 85

Executed Move: move_down
Reward Received: 0

## Step: 6
Current State:
 S  F  F  F  F  H  F 
 F  F  F  F [F] H  F 
 F  H  F  F  F  F  H 
 F  F  F  F  F  F  F 
 F  F  F  H  F  F  F 
 F  F  H  F  F  F  G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_down, Value: 78
- Step 2: Move: move_down, Value: 80

Executed Move: move_down
Reward Received: 0

## Step: 7
Current State:
 S  F  F  F  F  H  F 
 F  F  F  F  F  H  F 
 F  H  F  F [F] F  H 
 F  F  F  F  F  F  F 
 F  F  F  H  F  F  F 
 F  F  H  F  F  F  G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_down, Value: 85
- Step 2: Move: move_right, Value: 80

Executed Move: move_down
Reward Received: 0

## Step: 8
Current State:
 S  F  F  F  F  H  F 
 F  F  F  F  F  H  F 
 F  H  F  F  F  F  H 
 F  F  F  F [F] F  F 
 F  F  F  H  F  F  F 
 F  F  H  F  F  F  G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_right, Value: 85
- Step 2: Move: move_down, Value: 78

Executed Move: move_right
Reward Received: 0

## Step: 9
Current State:
 S  F  F  F  F  H  F 
 F  F  F  F  F  H  F 
 F  H  F  F  F  F  H 
 F  F  F  F  F [F] F 
 F  F  F  H  F  F  F 
 F  F  H  F  F  F  G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_down, Value: 85
- Step 2: Move: move_down, Value: 85

Executed Move: move_down
Reward Received: 0

## Step: 10
Current State:
 S  F  F  F  F  H  F 
 F  F  F  F  F  H  F 
 F  H  F  F  F  F  H 
 F  F  F  F  F  F  F 
 F  F  F  H  F [F] F 
 F  F  H  F  F  F  G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_right, Value: 85
- Step 2: Move: move_right, Value: 95

Executed Move: move_right
Reward Received: 0

## Step: 11
Current State:
 S  F  F  F  F  H  F 
 F  F  F  F  F  H  F 
 F  H  F  F  F  F  H 
 F  F  F  F  F  F  F 
 F  F  F  H  F  F  F 
 F  F  H  F  F [F] G 

Performing lookahead with depth=2 and sample_size=2...
Found best path:
- Step 1: Move: move_right, Value: 95

Executed Move: move_right
Reward Received: 1
Navigation was terminated.
Final State:
 S  F  F  F  F  H  F 
 F  F  F  F  F  H  F 
 F  H  F  F  F  F  H 
 F  F  F  F  F  F  F 
 F  F  F  H  F  F  F 
 F  F  H  F  F  F [G]
'''

In [ ]:
policyRefiner = PolicyRefiner(client, model, policyDb, optimizationPrompts)
policyRefiner.run(debugTrajectory3, debug=True)